# Datamine Studio RM - Holes3D Scripting Tutorial

This Jupyter Notebook demonstrates how to use the `dmstudio` Python package to automate Datamine processes via COM. 

The workflow is based on the **Holes3D Scripting Tutorial** from the official Datamine help documentation. It performs drillhole de-surveying (calculating 3D spatial coordinates) by combining and processing collars, assays, lithology, and survey data files.

## Workflow Overview
1. **Sort**: Sort all input files (`collars`, `assays`, `lithology`, `surveys`) on their key fields using `MGSORT`.
2. **Merge**: Merge the assay and lithology interval files using `HOLMER`.
3. **Join**: Relational join the collars file with the merged samples file using `JOIN`.
4. **De-survey**: Calculate 3D coordinates for all samples using `DESURV`.
5. **Display**: Export the final data as a CSV and view it using `pandas`.

In [1]:
# Step 1: Initialize connection to Datamine Studio RM
from dmstudio import dmcommands, initialize, special
import pandas as pd

# Connect to Studio RM
cmd = dmcommands.init(version='StudioRM')
oScript = initialize.studio('StudioRM')
print(f"Connected to: {oScript.Caption}")

pyrpa module not found, only available to SLR employees
Connected to: Studio RM 3.1.390.0 - Project.rmproj * - [3D]


In [2]:
# Step 2: Import local file mappings
# We import dmdir which contains variables pointing to the copied files in the project root.
import dmdir as f

print("Registered source files:")
print(f"  Collars: {f.__vb_collars_}")
print(f"  Assays: {f.__vb_assays_}")
print(f"  Lithology: {f.__vb_lithology_}")
print(f"  Surveys: {f.__vb_surveys_}")

Registered source files:
  Collars: _vb_collars
  Assays: _vb_assays
  Lithology: _vb_lithology
  Surveys: _vb_surveys


In [3]:
# Step 3: Sort the input tables on their respective key fields
print("Sorting collars by BHID...")
cmd.mgsort(in_i=f.__vb_collars_, out_o='tempcollar', keys_f=['BHID'])

print("Sorting assays by BHID and FROM...")
cmd.mgsort(in_i=f.__vb_assays_, out_o='tempassays', keys_f=['BHID', 'FROM'])

print("Sorting lithology by BHID and FROM...")
cmd.mgsort(in_i=f.__vb_lithology_, out_o='templith', keys_f=['BHID', 'FROM'])

print("Sorting surveys by BHID and AT...")
cmd.mgsort(in_i=f.__vb_surveys_, out_o='tempsurvey', keys_f=['BHID', 'AT'])

print("Sorting completed.")

Sorting collars by BHID...
Sorting assays by BHID and FROM...
Sorting lithology by BHID and FROM...
Sorting surveys by BHID and AT...
Sorting completed.


In [4]:
# Step 4: Merge assays and lithology tables using HOLMER
print("Merging assays and lithology (HOLMER)...")
cmd.holmer(
    inmods_i=['tempassays', 'templith'],
    out_o='temp1',
    bhid_f='BHID',
    from_f='FROM',
    to_f='TO'
)
print("Merge completed.")

Merging assays and lithology (HOLMER)...
Merge completed.


In [5]:
# Step 5: Join collars with merged sample intervals using JOIN
print("Relational joining collars and samples (JOIN)...")
# Fallback to run_command as keys_f parameter is not in the auto-generated wrapper's signature
cmd.run_command(
    "join &IN1=tempcollar &IN2=temp1 &OUT=temp2 *KEY1=BHID @SUBSETR=1 @SUBSETF=0 @CARTJOIN=0 @PRINT=0"
)
print("Join completed.")

Relational joining collars and samples (JOIN)...
Join completed.


In [6]:
# Step 6: De-survey the drillholes using DESURV to get 3D coordinates
print("De-surveying drillholes (DESURV)...")
cmd.desurv(
    inmods_i=['temp2', 'tempsurvey'],
    out_o='dholes',
    survsmth_p=1,
    print_p=0
)
print("De-survey completed.")

De-surveying drillholes (DESURV)...
De-survey completed.


In [7]:
# Step 7: View the final Data Definition
cmd.ddlist(in_i='dholes')

In [8]:
# Step 8: Export output to CSV and load in pandas
csv_filepath = 'dholes_output.csv'
oScript.ParseCommand(f'output &IN=dholes {csv_filepath} @CSV=1 @NODD=0')

# Read using pandas
df = pd.read_csv(csv_filepath)
print(f"Loaded {len(df)} rows from Datamine output.")
df.head(10)

Loaded 1044 rows from Datamine output.


,BHID,ENDDEPTH,REFSYS,REFMETH,HOLETYPE,ENDDATE,FROM,TO,AU,CU,DENSITY,LITH,NLITH,X,Y,Z,LENGTH,A0,B0,C0
0,VB2675,333.286896,Local,GPS,DD,11/11/98,0.000000,3.000000,NaN,NaN,NaN,Soil,0,6085.681150,5144.980629,186.403777,3.000000,180.289993,49.950001,0
1,VB2675,333.286896,Local,GPS,DD,11/11/98,2.500000,16.709127,NaN,NaN,NaN,Sandstone,1,6085.654756,5139.765767,180.199870,14.209127,180.289993,49.950001,0
2,VB2675,333.286896,Local,GPS,DD,11/11/98,16.709127,30.918254,NaN,NaN,NaN,Sandstone,1,6085.608480,5130.622939,169.323022,14.209127,180.289993,49.950001,0
3,VB2675,333.286896,Local,GPS,DD,11/11/98,30.918254,45.127380,NaN,NaN,NaN,Sandstone,1,6085.562205,5121.480110,158.446174,14.209127,180.289993,49.950001,0
4,VB2675,333.286896,Local,GPS,DD,11/11/98,45.127380,59.336507,NaN,NaN,NaN,Sandstone,1,6085.515930,5112.337281,147.569325,14.209127,180.289993,49.950001,0
5,VB2675,333.286896,Local,GPS,DD,11/11/98,59.336507,73.545634,NaN,NaN,NaN,Sandstone,1,6085.469654,5103.194453,136.692477,14.209127,180.289993,49.950001,0
6,VB2675,333.286896,Local,GPS,DD,11/11/98,73.545634,87.754761,NaN,NaN,NaN,Sandstone,1,6085.423379,5094.051624,125.815629,14.209127,180.289993,49.950001,0
7,VB2675,333.286896,Local,GPS,DD,11/11/98,87.754761,101.963888,NaN,NaN,NaN,Sandstone,1,6085.377104,5084.908795,114.938781,14.209127,180.289993,49.950001,0
8,VB2675,333.286896,Local,GPS,DD,11/11/98,101.963888,116.173014,NaN,NaN,NaN,Sandstone,1,6085.330828,5075.765966,104.061933,14.209127,180.289993,49.950001,0
9,VB2675,333.286896,Local,GPS,DD,11/11/98,116.173014,130.382141,NaN,NaN,NaN,Sandstone,1,6085.284553,5066.623138,93.185085,14.209127,180.289993,49.950001,0
